# Experiment 2: Coupling Reproduction

**Goal**: Evaluate how well each transfer method (DDSP, WORLD baseline)
reproduces the reference violin's pitch–timbre coupling.

For each method we compute:

1. **Coupling error** — $CE_i = \text{mean}\bigl((d_i^{\text{output}} - g_i^{\text{ref}}(f_0, \dot{f}_0, \ddot{f}_0))^2\bigr)$
2. **Own coupling model** — fit Model B on the output, compare coefficients
3. **Coefficient distance** — $\|\beta^{\text{output}} - \beta^{\text{ref}}\|_2$
4. **Correlation summary** — Pearson $r(d_i, f_0)$ per method vs reference

**Inputs**:
- Reference models from Experiment 1 (`exp1_ref_coupling_models.pkl`)
- Pre-extracted feature parquets for each method

**Outputs**: `artifacts/evaluation/exp2_*.csv` and figures

In [ ]:
import sys
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from evaluation.pitch_metrics import PitchMetrics
from visualize import (
    plot_coupling_scatter,
    plot_coupling_error_bars,
    plot_coefficient_distance_bars,
    plot_correlation_comparison,
)

print(f"Project root: {PROJECT_ROOT}")

## Configuration

In [ ]:
# ── Reference models from Experiment 1 ─────────────────────────────────
REF_MODEL_PATH = PROJECT_ROOT / "artifacts" / "evaluation" / "exp1_ref_coupling_models.pkl"

# ── Reference feature data (for scatter overlay) ───────────────────────
REF_PARQUET = PROJECT_ROOT / "data" / "processed" / "bach_violin_timbre_features.parquet"

# ── Transfer method feature data ──────────────────────────────────────
# Each entry: method_name -> path to pre-extracted parquet
# These parquets must contain the same descriptor columns + f0_hz + f0_confidence.
METHOD_PARQUETS = {
    "ddsp":     PROJECT_ROOT / "data" / "processed" / "ddsp_timbre_features.parquet",
    "baseline": PROJECT_ROOT / "data" / "processed" / "baseline_timbre_features.parquet",
}

# ── Output ─────────────────────────────────────────────────────────────
OUT_DIR = PROJECT_ROOT / "artifacts" / "evaluation"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Reference models: {REF_MODEL_PATH}  (exists: {REF_MODEL_PATH.exists()})")
for name, path in METHOD_PARQUETS.items():
    print(f"  {name}: {path}  (exists: {path.exists()})")

## 1. Load reference models and data

In [ ]:
with open(REF_MODEL_PATH, "rb") as fh:
    ref_bundle = pickle.load(fh)

ref_models   = ref_bundle["models"]
DESCRIPTORS  = ref_bundle["descriptors"]
cfg          = ref_bundle["config"]
ref_corr     = ref_bundle["correlation"]

dt             = cfg["dt"]
MEDIAN_WINDOW  = cfg["median_window"]
CONF_THRESH    = cfg["confidence_threshold"]

print(f"Descriptors: {DESCRIPTORS}")
print(f"Config: dt={dt}, median_window={MEDIAN_WINDOW}, conf_thresh={CONF_THRESH}")

# Load reference features for scatter overlay
df_ref = pd.read_parquet(REF_PARQUET)
ref_f0 = df_ref["f0_hz"].to_numpy(dtype=np.float64)
ref_f0[df_ref["f0_confidence"].to_numpy() < CONF_THRESH] = np.nan
ref_features_2d = df_ref[DESCRIPTORS].to_numpy(dtype=np.float64)
print(f"Reference: {len(df_ref):,} frames")

## 2. Load method features and compute derivatives

In [ ]:
method_data = {}  # name -> {f0, f0_dot, f0_ddot, features_2d}

for name, path in METHOD_PARQUETS.items():
    df = pd.read_parquet(path)
    f0 = df["f0_hz"].to_numpy(dtype=np.float64)
    f0[df["f0_confidence"].to_numpy() < CONF_THRESH] = np.nan
    f0_dot, f0_ddot = PitchMetrics.f0_dynamics(f0, dt=dt, median_window=MEDIAN_WINDOW)
    features_2d = df[DESCRIPTORS].to_numpy(dtype=np.float64)

    voiced = (f0 > 0) & np.isfinite(f0) & np.all(np.isfinite(features_2d), axis=1)

    method_data[name] = {
        "f0": f0,
        "f0_dot": f0_dot,
        "f0_ddot": f0_ddot,
        "features_2d": features_2d,
        "n_voiced": int(voiced.sum()),
    }
    print(f"{name}: {len(df):,} frames, {voiced.sum():,} voiced")

## 3. Coupling error

In [ ]:
all_ce = {}

for name, md in method_data.items():
    ce = PitchMetrics.coupling_error(
        md["features_2d"], md["f0"], md["f0_dot"], md["f0_ddot"],
        ref_models, DESCRIPTORS,
    )
    all_ce[name] = ce

df_ce = pd.DataFrame(all_ce).T
df_ce.index.name = "method"
print("Coupling Error (MSE):")
display(df_ce.round(4))

## 4. Fit each method's own coupling model

In [ ]:
method_models = {}

for name, md in method_data.items():
    models = PitchMetrics.fit_coupling_model(
        md["features_2d"], md["f0"], md["f0_dot"], md["f0_ddot"],
        DESCRIPTORS,
    )
    method_models[name] = models

    # Print R2 summary
    r2_rows = [{"descriptor": d, "R2_A": models[d]["r2_a"], "R2_B": models[d]["r2_b"]}
               for d in DESCRIPTORS]
    print(f"\n{name} - R\u00b2:")
    display(pd.DataFrame(r2_rows).set_index("descriptor").round(4))

## 5. Coefficient distance to reference

In [ ]:
all_cd = {}

for name, models in method_models.items():
    cd = PitchMetrics.coefficient_distance(models, ref_models, DESCRIPTORS)
    all_cd[name] = cd

df_cd = pd.DataFrame(all_cd).T
df_cd.index.name = "method"
print("Coefficient Distance (L2 norm):")
display(df_cd.round(4))

## 6. Correlation summary

In [ ]:
all_corr_r = {"reference": {d: ref_corr[d]["pearson_r"] for d in DESCRIPTORS}}

for name, md in method_data.items():
    corr = PitchMetrics.correlation_summary(md["features_2d"], md["f0"], DESCRIPTORS)
    all_corr_r[name] = {d: corr[d]["pearson_r"] for d in DESCRIPTORS}

df_corr = pd.DataFrame(all_corr_r).T
df_corr.index.name = "method"
print("Pearson r(descriptor, F0):")
display(df_corr.round(4))

## 7. Plots

In [ ]:
# 7.1 Scatter + regression overlay (reference + all methods)
scatter_data = {
    "reference": {
        "f0": ref_f0,
        "features_2d": ref_features_2d,
        "feat_keys": DESCRIPTORS,
        "models": ref_models,
    },
}
for name, md in method_data.items():
    scatter_data[name] = {
        "f0": md["f0"],
        "features_2d": md["features_2d"],
        "feat_keys": DESCRIPTORS,
        "models": method_models[name],
    }

plot_coupling_scatter(
    scatter_data, DESCRIPTORS,
    save_path=str(FIG_DIR / "exp2_coupling_scatter.png"),
)

In [ ]:
# 7.2 Coupling error bar chart
plot_coupling_error_bars(
    all_ce, descriptors=DESCRIPTORS,
    save_path=str(FIG_DIR / "exp2_coupling_error.png"),
)

In [ ]:
# 7.3 Coefficient distance bar chart
plot_coefficient_distance_bars(
    all_cd, descriptors=DESCRIPTORS,
    save_path=str(FIG_DIR / "exp2_coefficient_distance.png"),
)

In [ ]:
# 7.4 Correlation comparison
plot_correlation_comparison(
    all_corr_r, descriptors=DESCRIPTORS,
    save_path=str(FIG_DIR / "exp2_correlation_comparison.png"),
)

## 8. Save results

In [ ]:
df_ce.to_csv(OUT_DIR / "exp2_coupling_error.csv")
df_cd.to_csv(OUT_DIR / "exp2_coefficient_distance.csv")
df_corr.to_csv(OUT_DIR / "exp2_correlation_summary.csv")

# Save method models for Experiment 3
with open(OUT_DIR / "exp2_method_models.pkl", "wb") as fh:
    pickle.dump(method_models, fh)

print("Saved:")
for f in ["exp2_coupling_error.csv", "exp2_coefficient_distance.csv",
          "exp2_correlation_summary.csv", "exp2_method_models.pkl"]:
    print(f"  {OUT_DIR / f}")